In [28]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [29]:
device = torch.device("mps")

In [30]:
transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])
#Compose : sequence of transformations
#ToTensor: transform to tensor
#Normalize:normalize

In [31]:
train_data = datasets.MNIST(root='./data',train=True,download=True,transform=transform)

In [32]:
test_data = datasets.MNIST(root='./data',train=False,download=True,transform=transform)

In [33]:
batch_size=64
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True)
test_loader = DataLoader(train_data,batch_size=batch_size,shuffle=False)

In [34]:
class GDVarTester(nn.Module):
    def __init__(self):
        super(GDVarTester,self).__init__()
        self.fc1 = nn.Linear(28*28,128) # Input layer (28x28 flattened image to 128 neurons)
        self.fc2 = nn.Linear(128, 64) # Hidden layer
        self.fc3 = nn.Linear(64, 10) # Output layer (10 classes for digits 0-9)
        self.softmax = nn.Softmax(dim=1) # Softmax for classification

    def forward(self,x):
        x = x.view(-1,28*28) #flatten the image
        x = torch.relu(self.fc1(x)) # First hidden layer with ReLU
        x = torch.relu(self.fc2(x)) # Second hidden layer with ReLU
        x = self.fc3(x) # Output layer
        return x
    
model = GDVarTester()

In [35]:
criterion = nn.CrossEntropyLoss()

In [36]:
optimizer = optim.SGD(model.parameters(),lr=0.0001)

In [39]:
def train_model(train_loader,optimizer_type='SGD',num_epochs=15):
    model = GDVarTester()
    model.to(device)
    criterion=nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(),lr=0.001)
    
    train_losses=[]

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs,labels in train_loader:
            inputs,labels = inputs.to(device),labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss=criterion(outputs,labels)
            loss.backward()
            optimizer.step()

            running_loss+=loss.item()

            #Accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total
        train_losses.append(avg_loss)


        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {train_accuracy:.2f}%")
    return model, train_losses, train_accuracy

In [40]:
# Example setup (you must define train_data and device before this)
batch_sizes = [len(train_data), 64, 1]  # Batch GD, Mini-batch GD, SGD
methods = ['Batch GD', 'Mini-batch GD', 'SGD']

results = {}
accuracies = {}

for method, batch_size in zip(methods, batch_sizes):
    print(f"\nTraining with {method} (batch size = {batch_size})...")
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    model, train_losses, train_accuracy = train_model(train_loader, optimizer_type=method)
    results[method] = train_losses
    accuracies[method] = train_accuracy



Training with Batch GD (batch size = 60000)...
Epoch [1/15], Loss: 2.3100, Accuracy: 8.54%
Epoch [2/15], Loss: 2.3098, Accuracy: 8.55%
Epoch [3/15], Loss: 2.3096, Accuracy: 8.58%
Epoch [4/15], Loss: 2.3094, Accuracy: 8.58%
Epoch [5/15], Loss: 2.3092, Accuracy: 8.60%
Epoch [6/15], Loss: 2.3090, Accuracy: 8.62%
Epoch [7/15], Loss: 2.3088, Accuracy: 8.64%
Epoch [8/15], Loss: 2.3086, Accuracy: 8.65%
Epoch [9/15], Loss: 2.3084, Accuracy: 8.67%
Epoch [10/15], Loss: 2.3082, Accuracy: 8.68%
Epoch [11/15], Loss: 2.3080, Accuracy: 8.70%
Epoch [12/15], Loss: 2.3078, Accuracy: 8.71%
Epoch [13/15], Loss: 2.3076, Accuracy: 8.73%
Epoch [14/15], Loss: 2.3074, Accuracy: 8.76%
Epoch [15/15], Loss: 2.3072, Accuracy: 8.77%

Training with Mini-batch GD (batch size = 64)...
Epoch [1/15], Loss: 2.2241, Accuracy: 24.37%
Epoch [2/15], Loss: 1.9741, Accuracy: 51.35%
Epoch [3/15], Loss: 1.5232, Accuracy: 68.10%
Epoch [4/15], Loss: 1.0808, Accuracy: 75.43%
Epoch [5/15], Loss: 0.8180, Accuracy: 80.08%
Epoch [6/15